# 07 · Recursion & Backtracking

**Recursion** = a function calling itself on a smaller problem. **Backtracking** = try a choice, explore, then **undo** it to try the next one.

**Pick / not-pick pattern:** at every index there are two choices: **pick** `nums[index]` (add it, recurse on `index+1`) or **not pick** (undo the add, recurse on `index+1`). This builds a recursion tree with `2ⁿ` leaves.

The code is exactly my original; I added explanations and comments.

**Topics:** power set · subsequences with sum `k` (all / one / count) · binary strings without consecutive 1s

## 1. Power set with a bitmask
Every number `0 … 2ⁿ−1` is a mask over the elements (see the Bit Manipulation notebook).

In [ ]:
# power set
nums=[1,2,3]
n=len(nums)
# 2**n subsets
total_subsets=1<<n
result=[]
for num in range(0,total_subsets):
  list=[]
  for i in range(0,n):
    # bit i set → include nums[i]
    if num &(1<<i) !=0:
      list.append(nums[i])
  result.append(list)
print(result)


[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


## 2. Power set with recursion
At each `index` we first **pick** the element, recurse, then **backtrack** (`pop`) and recurse again for **not pick**. At `index >= len(nums)` every element has been decided, so the current subset is stored. `subset.copy()` is essential: `subset` is one list that keeps changing, so storing it directly would leave empty lists at the end.

The output order is `[1,2,3], [1,2], [1,3], [1], …` because "pick" is explored first.

In [ ]:
def func(index,subset):
  # base case: every element decided
  if index >= len(nums):
    # store a COPY of the current subset
    result.append(subset.copy())
    return
  # choice 1: pick
  subset.append(nums[index])
  # recurse with the element included
  func(index+1,subset)
  # backtrack: undo the pick
  subset.pop()
  # choice 2: not pick (recurse without the element)
  func(index+1,subset)
nums=[1,2,3]
result=[]
func(0,[])
print(result)


[[1, 2, 3], [1, 2], [1, 3], [1], [2, 3], [2], [3], []]


## 3. All subsequences with sum `k`
Carry a running `total`:
- `total == k` → store the subset and stop
- `total > k` → stop early (**pruning**, valid because the numbers are positive)
- `index` out of range → stop

For `[2,3,6,4,7]`, `k = 7` the answers are `[3,4]` and `[7]`. *(The lines using the variable `sum` are redundant: `sum=sum-e` is never used afterwards, and the name hides Python's `sum()`.)*

In [ ]:
def back_track(index,total,subset):
  # found a subsequence with the required sum
  if total ==k:
    result.append(subset.copy())
    return
  # prune: adding more positive numbers can only increase the sum
  elif total > k:
    return
  if index >=len(nums):
    return
  # pick
  subset.append(nums[index])
  sum=total+nums[index]
  # explore with the element included
  back_track(index+1,sum,subset)
  # backtrack
  e=subset.pop()
  sum=sum-e
  # not pick
  back_track(index+1,total,subset)
nums=[2,3,6,4,7]
k=7
result=[]
back_track(0,0,[])
print(result)

[[3, 4], [7]]


## 4. Print ONE subsequence with sum `k`
Once one answer is found there is no need to search further. The function **returns `True`** when it finds one and each caller passes that `True` upward. The subset is popped only if the pick branch **fails**.

In [ ]:
def subsequence_with_sum(index,total,subset)->bool:
  if total ==k:
    result.append(subset.copy())
    # found one → tell the callers to stop
    return True
  elif total > k:
    return False
  if index >=len(nums):
    return False
  subset.append(nums[index])
  sum=total+nums[index]
  # try the pick branch first
  pick=subsequence_with_sum(index+1,sum,subset)
  # success in the pick branch → stop immediately
  if pick ==True:
    return True
  # pick failed → undo it
  subset.pop()
  sum=total
  not_pick =subsequence_with_sum(index+1,total,subset)
  # otherwise the answer is whatever the not-pick branch finds
  return not_pick
nums=[2,3,6,4,7]
k=11
result=[]
print(subsequence_with_sum(0,0,[]))



True


## 5. COUNT subsequences with sum `k`
We only need the number of subsequences, so the function **returns a number**: `pick + not_pick`. A valid sum returns `1`, an invalid branch returns `0`. No list is needed.

In [ ]:
def no_of_subsequence_with_sum_k(index,total):
  if total ==k:

    # one valid subsequence found
    return 1
  elif total > k:
    # this branch cannot give the sum
    return 0
  if index >=len(nums):
    return 0

  sum=total+nums[index]
  pick=no_of_subsequence_with_sum_k(index+1,sum)

  sum=total
  not_pick=no_of_subsequence_with_sum_k(index+1,sum)
  # total ways = ways with the element + ways without it
  return pick +not_pick
nums=[2,3,6,4,7]
k=9
print(no_of_subsequence_with_sum_k(0,0))

3


## 6. Binary strings of length `n` with no consecutive 1s
The problem statement is in the next cell (a comment). ⚠️ The output shown under that cell, `{1, 2, 4, 6}`, is a **stale output** from some earlier cell; the comment itself prints nothing.

In [ ]:
#generate all the binary string
#Given an integer N , Print all binary strings of size N which do not contain consecutive 1s.

#A binary string is that string which contains only 0 and 1.



{1, 2, 4, 6}


Build the string position by position: a `0` is **always** allowed; a `1` is allowed only if the previous character was **not** `1`. The `flag` carries that information (`True` after a 0, `False` after a 1). After trying `1` we put `0` back (backtrack). For `n = 4` there are 8 strings (Fibonacci-like growth).

In [ ]:
class solution:
  def solve(self,index,flag,numbers,result):
    if index >=len(numbers):
      result.append(''.join(numbers))
      return
    # option 1: place 0 (always allowed)
    numbers[index]='0'
    # after a 0 the next position may be 1
    self.solve(index+1,True,numbers,result)
    # a 1 may be placed only if the previous character was not 1
    if flag ==True:
      numbers[index]="1"
      # after a 1 the next position cannot be 1
      self.solve(index+1,False,numbers,result)
      # backtrack
      numbers[index]="0"
  def generate_binary_strings(self,n):
    numbers=["0"]*n
    result=[]
    self.solve(0,True,numbers,result)
    return result
s=solution()
print(s.generate_binary_strings(4))

['0000', '0001', '0010', '0100', '0101', '1000', '1001', '1010']


---
### ✅ Key takeaways
- **Pick / not-pick** is the template for subsets, subsequences and combinations.
- **Backtrack** = undo your change (`pop`) before trying the alternative.
- Store `subset.copy()`, not `subset`.
- Return `True/False` for "find one", a **number** for "count", append to a **list** for "find all".